# TurboVec RAG on Snowpark Container Services

Multi-tenant RAG demo: embed, index, search, generate — all within Snowflake.

In [ ]:
import requests, os

TURBOVEC_ENDPOINT = 'http://turbovec:8000'
print(requests.get(f'{TURBOVEC_ENDPOINT}/health').json())

In [ ]:
import snowflake.connector
conn = snowflake.connector.connect(connection_name=os.getenv('SNOWFLAKE_CONNECTION_NAME','default'))
cur = conn.cursor()

def embed_text(text):
    cur.execute(f"SELECT AI_EMBED('snowflake-arctic-embed-m-v1.5', $${text}$$)::ARRAY")
    return cur.fetchone()[0]

In [ ]:
docs = {
    'tenant_a': [
        'TurboQuant is a data-oblivious quantization algorithm from Google Research.',
        'It achieves near-optimal distortion without codebook training.',
        'TurboVec implements TurboQuant in Rust with Python bindings.',
        '16x compression is achieved at 2-bit quantization for d=1536 vectors.',
        'Filtered search uses allowlists to restrict results to specific tenants.',
    ],
    'tenant_b': [
        'Snowpark Container Services runs Docker containers inside Snowflake.',
        'SPCS supports CPU and GPU compute pools for different workloads.',
        'Cortex AI functions provide embedding and LLM capabilities.',
        'AI_EMBED generates vector embeddings from text in Snowflake.',
        'Multi-tenant isolation is critical for enterprise deployments.',
    ],
}
embeddings = {}
for tid, d in docs.items():
    embeddings[tid] = [embed_text(t) for t in d]
    print(f'Embedded {len(d)} docs for {tid}')

## Load vectors into TurboVec

In [ ]:
doc_id = 0
for tid, vecs in embeddings.items():
    ids = list(range(doc_id, doc_id + len(vecs)))
    r = requests.post(f'{TURBOVEC_ENDPOINT}/add', json={'vectors': vecs, 'ids': ids, 'tenant_id': tid})
    print(f'{tid}: {r.json()}')
    doc_id += len(vecs)

## Multi-tenant search (tenant isolation at SIMD kernel level)

In [ ]:
query = 'How does vector compression work?'
qvec = embed_text(query)

all_docs = docs['tenant_a'] + docs['tenant_b']
for tid in ['tenant_a', 'tenant_b']:
    r = requests.post(f'{TURBOVEC_ENDPOINT}/search', json={'query': qvec, 'k': 3, 'tenant_id': tid}).json()
    print(f'\n{tid} ({r["latency_ms"]}ms):')
    for s, i in zip(r['scores'], r['ids']):
        print(f'  [{s:.4f}] {all_docs[i]}')

## RAG with AI_COMPLETE

In [ ]:
def rag(question, tenant_id, k=3):
    qvec = embed_text(question)
    r = requests.post(f'{TURBOVEC_ENDPOINT}/search', json={'query': qvec, 'k': k, 'tenant_id': tenant_id}).json()
    context = '\n'.join(docs[tenant_id][i] for i in r['ids'] if i < len(docs[tenant_id]))
    prompt = f'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
    cur.execute(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', $${prompt}$$)")
    return cur.fetchone()[0]

print(rag('What compression ratio does TurboVec achieve?', 'tenant_a'))

## Telemetry (per-query cost attribution)

In [ ]:
for r in requests.get(f'{TURBOVEC_ENDPOINT}/telemetry?limit=10').json():
    print(f"tenant={r.get('tenant_id')}, latency={r['latency_us']}us, k={r['k_returned']}")